# 07 - Run a CPU-conscious DNABERT2 embedding smoke test

The grouped-test character-k-mer baseline is now available. Before scaling transformer inference, this notebook prepares a PyTorch-fallback DNABERT2 snapshot and embeds only four isolates using 16 windows per isolate. This validates model download, tokenizer loading, MPS inference, embedding export, and the embedding-baseline script.

## 1. Configuration

Transformer inference uses Apple MPS on this machine. DNABERT2's official code treats Triton flash attention as optional, but recent Transformers dependency scanning rejects the optional Triton module before fallback can occur. The preparation step creates a local copy that uses DNABERT2's ordinary PyTorch attention fallback. Execution switches default to `False`. The initial model download is approximately 469 MB.

In [ ]:
from __future__ import annotations

from pathlib import Path
import shutil
import subprocess
import sys

import pandas as pd
import torch
import transformers

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

WINDOWS_PATH = PROJECT_ROOT / "data" / "processed" / "smoke_windows.csv"
SPLITS_PATH = PROJECT_ROOT / "data" / "processed" / "isolate_splits.csv"
EMBEDDINGS_PATH = PROJECT_ROOT / "data" / "processed" / "dnabert2_smoke_embeddings.npz"
METRICS_PATH = PROJECT_ROOT / "reports" / "dnabert2_smoke_embedding_baseline.json"
MODEL_DIR = PROJECT_ROOT / "data" / "external" / "dnabert2_pytorch_fallback"
LIMIT_ISOLATES = 4
MAXIMUM_WINDOWS_PER_ISOLATE = 16
BATCH_SIZE = 1
PREPARE_DNABERT2_FALLBACK = False
RUN_DNABERT2_SMOKE = False
RUN_EMBEDDING_BASELINE = False

print("Project root:", PROJECT_ROOT)

## 2. Review the local environment and workload

In [ ]:
windows = pd.read_csv(WINDOWS_PATH)
selected_ids = windows["isolate_id"].drop_duplicates().head(LIMIT_ISOLATES)
selected = windows[windows["isolate_id"].isin(selected_ids)].groupby("isolate_id", group_keys=False).head(MAXIMUM_WINDOWS_PER_ISOLATE)

if torch.cuda.is_available():
    device = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Device:", device)
print("Free disk space: %.2f GB" % (shutil.disk_usage(PROJECT_ROOT).free / 1_000_000_000))
print("Smoke isolates:", selected["isolate_id"].nunique())
print("Smoke windows:", len(selected))
selected.groupby(["isolate_id", "label"]).size().rename("windows")

## 3. Prepare the PyTorch-fallback DNABERT2 snapshot

Change `PREPARE_DNABERT2_FALLBACK` to `True` once. This downloads the official model snapshot and disables only the optional Triton import.

In [ ]:
prepare_command = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "prepare_dnabert2_pytorch_fallback.py"),
    "--output-dir",
    str(MODEL_DIR),
]
print(" ".join(prepare_command))
if PREPARE_DNABERT2_FALLBACK:
    subprocess.run(prepare_command, cwd=PROJECT_ROOT, check=True)
print("Prepared model exists:", MODEL_DIR.exists())

## 4. Extract frozen DNABERT2 embeddings

Change `RUN_DNABERT2_SMOKE` to `True` after preparing the model snapshot. Start with batch size 1.

In [ ]:
embedding_command = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "embed_dnabert2.py"),
    "--windows",
    str(WINDOWS_PATH),
    "--output",
    str(EMBEDDINGS_PATH),
    "--model",
    str(MODEL_DIR),
    "--limit-isolates",
    str(LIMIT_ISOLATES),
    "--maximum-windows-per-isolate",
    str(MAXIMUM_WINDOWS_PER_ISOLATE),
    "--batch-size",
    str(BATCH_SIZE),
]
print(" ".join(embedding_command))
if RUN_DNABERT2_SMOKE:
    subprocess.run(embedding_command, cwd=PROJECT_ROOT, check=True)
print("Embeddings exist:", EMBEDDINGS_PATH.exists())

## 5. Run the frozen-embedding technical baseline

Change `RUN_EMBEDDING_BASELINE` to `True` only after the embeddings exist. Metrics from four isolates are a pipeline check, not a scientific result.

In [ ]:
baseline_command = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "train_embedding_baseline.py"),
    "--embeddings",
    str(EMBEDDINGS_PATH),
    "--splits",
    str(SPLITS_PATH),
    "--output",
    str(METRICS_PATH),
]
print(" ".join(baseline_command))
if RUN_EMBEDDING_BASELINE:
    subprocess.run(baseline_command, cwd=PROJECT_ROOT, check=True)
if METRICS_PATH.exists():
    print(METRICS_PATH.read_text())

## 6. Next step

If CPU inference is tolerable, generate a capped full-cohort DNABERT2 window plan and resume embeddings incrementally. If it is too slow, run the embedding stage in Colab or another GPU environment while keeping the same manifests and grouped splits.